# Lesson 27 — Datetime: Practice (SUPPORTING)

Work in `lessons/lesson-27-datetime/` (point a terminal at that folder).

Before you start, read `lesson.md` — every construct used in these exercises is
taught there. Write each deliverable in its stated file (R16 — no
`solution.py`, no double-write), then run each **Driver:** cell. Every
`Expected` block must match byte-exact.

You create: `tenure.py` (E1–M2) · `aging.py` + `report.py` (Challenge,
which imports `tenure.py` — write once, reuse).
Provided: `buggy_tenure.py` (buggy — true hunt, you fix it in place).
Installs: none — stdlib only (`datetime` taught in lesson.md).


## E1 — Round-trip dates without loss

## Intent
**Parse and format canonically — the round trip proves the shape.**

**Type:** REAL-WORLD — Easy — writes 1 file

**Task**
In `tenure.py`, define `parse_iso(text: str) -> date` (strict
`%Y-%m-%d`; any failure raises `ValueError("invalid date: bad value")`)
and `show(day: date) -> str` (`strftime("%d %b %Y")`, e.g.
`"05 Jan 2026"`).

**File tree**

- `tenure.py` (`parse_iso`, `show`)

**Interface contract**

- `parse_iso("2026-01-05") == date(2026, 1, 5)`.
- `show(date(2026, 1, 5)) == "05 Jan 2026"`.

**Constraints**

- **One parse format (`%Y-%m-%d`), stated once.**
- **Exact error string verbatim.**

**Coverage**

- Parse/format → lesson.md Core Concepts 1/4.

**Expected** (Driver cell):
```text
E1 OK
```


In [ ]:
from datetime import date
import tenure as t

assert t.parse_iso("2026-01-05") == date(2026, 1, 5)
assert t.show(date(2026, 1, 5)) == "05 Jan 2026"
assert t.parse_iso(t.parse_iso("2024-02-29").isoformat()) == date(2024, 2, 29)
for bad in ["2026-13-99", "01/05/2026", "not-a-date", ""]:
    try:
        t.parse_iso(bad)
    except ValueError as e:
        assert str(e) == "invalid date: bad value", (bad, str(e))
    else:
        raise AssertionError(("no raise", bad))
print("E1 OK")

## E2 — Tenures in days, shifts on the calendar

## Intent
**Compute durations on objects — February included.**

**Type:** REAL-WORLD — Easy — writes 1 file

**Task**
In `tenure.py`, define `tenure_days(start: date, end: date) -> int`
(`(end - start).days`, negative allowed) and `shift(day: date,
delta_days: int) -> date` (`day + timedelta(days=delta_days)`).

**File tree**

- `tenure.py` (`tenure_days`, `shift`)

**Interface contract**

- `tenure_days(date(2026, 1, 5), date(2026, 3, 1)) == 55`.
- `shift(date(2026, 1, 5), 30) == date(2026, 2, 4)`.

**Constraints**

- **No string work anywhere (objects in, objects out).**
- **Leap spans must hold (driver includes Feb 2024).**

**Coverage**

- Object arithmetic → lesson.md Core Concept 3.

**Expected** (Driver cell):
```text
E2 OK
```


In [ ]:
from datetime import date
import tenure as t

assert t.tenure_days(date(2026, 1, 5), date(2026, 3, 1)) == 55
assert t.tenure_days(date(2024, 2, 1), date(2024, 3, 1)) == 29
assert t.tenure_days(date(2026, 3, 1), date(2026, 1, 5)) == -55
assert t.shift(date(2026, 1, 5), 30) == date(2026, 2, 4)
assert t.shift(date(2024, 2, 28), 1) == date(2024, 2, 29)
print("E2 OK")

## E3 — The canonical gate

## Intent
**Accept canonical dates, reject everything else — never normalize.**

**Type:** REAL-WORLD — Easy — writes 1 file

**Task**
In `tenure.py`, define `parse_date(text: str) -> date`: parse with
`%Y-%m-%d` (failure → `ValueError("invalid date: bad value")`), then
the round-trip check (`isoformat() != text` →
`ValueError("invalid date: not canonical")`). Parseable-but-drifted
input (`"2026-1-5"`) is rot, not a date.

**File tree**

- `tenure.py` (`parse_date`)

**Interface contract**

- `parse_date("2026-01-05") == date(2026, 1, 5)`.

**Constraints**

- **Check order: value first, canonical second.**
- **Exact messages verbatim, in that order.**

**Coverage**

- Reject-don't-normalize → lesson.md Core Concept 2 (+ WP2 retrieval).

**Expected** (Driver cell):
```text
E3 OK
```


In [ ]:
from datetime import date
import tenure as t

assert t.parse_date("2026-01-05") == date(2026, 1, 5)
for bad, msg in [("2026-13-99", "invalid date: bad value"),
                 ("not-a-date", "invalid date: bad value"),
                 ("2026-1-5", "invalid date: not canonical"),
                 ("2026-01-5", "invalid date: not canonical")]:
    try:
        t.parse_date(bad)
    except ValueError as e:
        assert str(e) == msg, (bad, str(e))
    else:
        raise AssertionError(("no raise", bad))
print("E3 OK")

## M1 — Classify tenures into buckets

## Intent
**Turn signup dates into aged buckets — parsed, computed, labeled.**

**Type:** REAL-WORLD — Medium — writes 1 file

**Task**
In `tenure.py`, define `bucket(signup: str, today: date) -> str` from
YOUR `parse_date` (this reuse is the exercise — no reimplemented
parsing). Tenure under 30 days → `"new"`; 30–364 → `"active"`;
365+ → `"veteran"`. Future signups (negative tenure) raise
`ValueError("invalid tenure: future signup")`.

**File tree**

- `tenure.py` (`bucket` — no new file)

**Interface contract**

- `bucket("2026-01-05", date(2026, 3, 1)) == "active"` (55 days).

**Constraints**

- **Reuse your own gate (no second parser).**
- **Boundaries exact: 30 and 365 belong upward.**
- **Exact error string verbatim.**

**Coverage**

- Learner composition (parse → compute → label) → Core Concepts 2–3.

**Expected** (Driver cell):
```text
M1 OK
```


In [ ]:
from datetime import date
import tenure as t

assert t.bucket("2026-01-05", date(2026, 3, 1)) == "active"
assert t.bucket("2026-02-15", date(2026, 3, 1)) == "new"
assert t.bucket("2024-01-01", date(2026, 3, 1)) == "veteran"
assert t.bucket("2026-02-01", date(2026, 3, 3)) == "active"
assert t.bucket("2025-03-01", date(2026, 3, 1)) == "veteran"
try:
    t.bucket("2026-04-01", date(2026, 3, 1))
except ValueError as e:
    assert str(e) == "invalid tenure: future signup"
else:
    raise AssertionError("no raise")
try:
    t.bucket("2026-1-5", date(2026, 3, 1))
except ValueError as e:
    assert str(e) == "invalid date: not canonical"
else:
    raise AssertionError("no raise")
print("M1 OK")

## M2 — Compose the aging pipeline yourself

## Intent
**Build the month-bucketed aging report from your own pieces — order observable.**

**Type:** REAL-WORLD — Medium — writes 1 file

**Task**
In `tenure.py`, define `age_report(signups, today)` from YOUR
`parse_date` (composition is the exercise — driver checks behavior
only). Parse each signup; bucket by canonical month (`isoformat()[:7]`
— legitimate here: output of the parser, guaranteed canonical, never
raw strings); count per month in first-seen order; rotten lines raise
the gate error (nothing skipped silently). Return the ordered
`(month, count)` pair list.

**File tree**

- `tenure.py` (`age_report` — no new file)

**Constraints**

- **Reuse your own gate (no reimplemented parsing).**
- **Month key from parser output only (`isoformat()[:7]`), never
  from raw strings.**
- **First-seen month order; rotten input raises the gate error.**

**Coverage**

- Learner-authored composition → M2 law + Core Concepts 1–3.

**Expected** (Driver cell):
```text
M2 OK
```


In [ ]:
from datetime import date
import tenure as t

got = t.age_report(["2026-01-05", "2026-01-20", "2026-02-01"], date(2026, 3, 1))
assert got == [("2026-01", 2), ("2026-02", 1)], got
assert t.age_report([], date(2026, 3, 1)) == []
try:
    t.age_report(["2026-1-5"], date(2026, 3, 1))
except ValueError as e:
    assert str(e) == "invalid date: not canonical"
else:
    raise AssertionError("no raise")
print("M2 OK")

## H1 — Hunt: four date bugs in buggy_tenure

## Intent
**Diagnose four real datetime failures in a provided module — fix each where it lives.**

**Type:** DEBUG — Hard — fixes 1 file

**Task**
`buggy_tenure.py` is provided, buggy: four planted failures, one per
Pitfall (slice-order, leap blindness, format swap, normalizing rot).
Nothing else in the file is wrong. Run the DEBUGGING protocol:
reproduce via the driver, read the traceback, locate, hypothesize, fix,
explain. Fix in place; do not restructure the module.

**File tree**

- `buggy_tenure.py` (`month_of`, `feb_days`, `parse_us`, `normalize` —
  YOU fix all four)

**Interface contract** (fixed states, exact)

- `month_of` buckets by parsed month, never by raw slice.
- `feb_days` counts February exactly across leap and common years.
- `parse_us` reads US `MM/DD/YYYY` with the fields in order.
- `normalize` rejects non-canonical input (never reformats rot).

**Constraints**

- **Four fixes only — no refactoring, no new symbols.**
- **Each fix stays in its own layer.**
- **Fill one 7-row chain per bug in the next cell, as you fix.**

**Coverage**

- The four date failures → lesson.md Pitfalls 1–4.

**Expected** (Driver cell):
```text
H1 OK
```


### H1 chain reports (one per bug — fill as you fix)

#### Bug 1 — month_of slice
| Step | Content |
|---|---|
| Symptom | ----> 7 assert bt.month_of("2026-1-5") == "2026-01" |
| Trigger | month_of() |
| Hypothesis | parsed incorrectly |
| Evidence | the result on month 01 and 1 is inconsistent |
| Cause | no parse at all |
| Fix | add parse |
| Proof | green |

#### Bug 2 — feb_days leap
| Step | Content |
|---|---|
| Symptom | ---> 10 assert bt.feb_days(1900) == 28 |
| Trigger | feb_days |
| Hypothesis | uncaught leap year |
| Evidence | 1900 feb is not leap year |
| Cause | manual checking for leap year |
| Fix | use date as counter |
| Proof | green |

#### Bug 3 — parse_us swap
| Step | Content |
|---|---|
| Symptom | ValueError: time data '05/13/2026' does not match format '%Y-%d-%m' |
| Trigger | parse_us |
| Hypothesis | wrong format |
| Evidence | y-d-m instead of m/d/y |
| Cause | wrong format |
| Fix | change it to %m/%d/%Y |
| Proof | green |

#### Bug 4 — normalize rot
| Step | Content |
|---|---|
| Symptom | AssertionError: no raise |
| Trigger | normalize |
| Hypothesis | no exception handling and canonical validation |
| Evidence | no error raised on normalize |
| Cause | no handler |
| Fix | add exception handling and canonical validation |
| Proof | green |


In [ ]:
import importlib
from datetime import date
import buggy_tenure as bt
importlib.reload(bt)

assert bt.month_of("2026-01-05") == "2026-01"
assert bt.month_of("2026-1-5") == "2026-01"
assert bt.feb_days(2024) == 29
assert bt.feb_days(2026) == 28
assert bt.feb_days(1900) == 28
assert bt.parse_us("05/13/2026") == date(2026, 5, 13)
assert bt.parse_us("12/25/2025") == date(2025, 12, 25)
assert bt.normalize("2026-01-05") == "2026-01-05"
try:
    bt.normalize("2026-1-5")
except ValueError as e:
    assert str(e) == "invalid date: not canonical"
else:
    raise AssertionError("no raise")
print("H1 OK")

## Challenge — Dawn tenure run over signups

## Intent
**Ship a tenure run over the signup log: every account aged, every rotten line skipped loudly — computed on objects end to end.**

**Type:** REAL-WORLD — Micro Build — writes 2 files

**Task**
The billing desk runs one command before dawn:
`python report.py <signups.csv> <tenure.txt>`. It parses each signup
(`name,YYYY-MM-DD`), ages every account against 2026-03-01, buckets by
canonical month with counts, skips rotten lines with a loud `skip`
line, and writes a tenure report that is also printed — same bytes
both places.

Create `aging.py` and `report.py` in this folder. Reuse `tenure.py`
(`parse_date` — write once, import, never double-write).

**Flow** (the whole wiring — every arrow is required)

```text
main reads <signups.csv> as text lines (always a valid text file; small log, read whole)
  → per (i, raw) in file order, split once on the first comma:
      name, text = halves (missing comma → skip line, gate message)
      day = parse_date(text)               # rotten → skip line, no aging
      tenure = (TODAY - day).days          # TODAY = date(2026, 3, 1), objects only
      bucket: <30 new / 30–364 active / 365+ veteran
      future (negative) → skip line with the tenure message
  → text = build_aging(months, lines)
  → write text to <tenure.txt> AND print the same text to stdout
```

The log is always a valid text file — no corrupt-file handling required.
"Missing log" means the path does not exist or is not a file. CSV rows
use the first comma only (names never contain commas in the fixture).

**File tree**

- `aging.py` (`build_aging`)
- `report.py` (`main`)

**Interface contract**

- `aging.build_aging(months, lines)` returns header + month rows +
  account lines + trailing newline (each section skipped when empty).
  It performs no I/O.
- Account lines: `line {i}: {name} {bucket} {tenure}` (file index,
  then name, bucket, day count) and `skip {i}: {exact message}`.
- Header shape: `aged A, skipped S` (A = aged accounts, S = skip
  lines). Month rows: `month YYYY-MM: C` in first-seen order.
- `report.main(argv)` — house shape (L20–L26): full `sys.argv`
  (`main(sys.argv)` guard), paths at `argv[1]` / `argv[2]`, `-> None`,
  errors leave via `sys.exit(1)` after one stderr line.

**Constraints**

- **Per line: split → parse → age → bucket; rotten → skip line with
  the exact gate/tenure message.**
- **No string arithmetic on dates anywhere (objects only).**
- **Wrong argc prints `usage: report.py <signups.csv> <tenure.txt>`
  to stderr and exits 1.**
- **Missing log prints `bad log: not found` to stderr and exits 1.**
- **Stdout and the tenure file are byte-identical.**

**Coverage**

- Dates parsed + computed + formatted → lesson.md Core Concepts 1–6
  + house entry (L20–L26 retrieval).

**Expected** (Driver cell, real subprocess runs, fixture of 5):
```text
CH OK
```


In [ ]:
import subprocess, sys, tempfile, pathlib
HOME = pathlib.Path.cwd()
rows = ["ann,2026-01-05", "bob,2026-1-5", "cid,2025-02-15", "dex,2026-13-99", "eli,2026-02-28"]
expected = ("aged 3, skipped 2\n" "month 2026-01: 1\n" "month 2025-02: 1\n"
    "month 2026-02: 1\n" "line 0: ann active 55\n"
    "skip 1: invalid date: not canonical\n" "line 2: cid veteran 379\n"
    "skip 3: invalid date: bad value\n" "line 4: eli new 1\n")
with tempfile.TemporaryDirectory() as td:
    td = pathlib.Path(td)
    lg, rp = td / "signups.csv", td / "tenure.txt"
    lg.write_text("\n".join(rows) + "\n", encoding="utf-8")
    run = subprocess.run(
        [sys.executable, "report.py", str(lg), str(rp)],
        capture_output=True, text=True, cwd=HOME)
    assert run.returncode == 0, run.stderr
    assert run.stdout == expected, repr(run.stdout)
    assert rp.read_text(encoding="utf-8") == expected
    usage = subprocess.run(
        [sys.executable, "report.py", str(lg)],
        capture_output=True, text=True, cwd=HOME)
    assert usage.returncode == 1
    assert usage.stderr == "usage: report.py <signups.csv> <tenure.txt>\n"
    missing = subprocess.run(
        [sys.executable, "report.py", str(td / "nope.csv"), str(rp)],
        capture_output=True, text=True, cwd=HOME)
    assert missing.returncode == 1
    assert missing.stderr == "bad log: not found\n"
print("CH OK")